In [1]:
import pandas as pd
import sklearn
print("tudo certo", pd.__version__, sklearn.__version__)


tudo certo 3.0.6 1.9.1


In [2]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

df = pd.read_excel("dados/FakeRecogna-master/dataset/FakeRecogna.xlsx")
print(df.shape)
print(df.columns.tolist())
df.head()

(11903, 8)
['Titulo', 'Subtitulo', 'Noticia', 'Categoria', 'Data', 'Autor', 'URL', 'Classe']


,Titulo,Subtitulo,Noticia,Categoria,Data,Autor,URL,Classe
0,\n\nPapa Francisco foi preso sob acusação de t...,Boato – Ocorreu um apagão no Vaticano. O papa ...,apagão vaticano papar presar acusação tráfico ...,entretenimento,11/01/2021,\nEdgard Matsuki,https://www.boatos.org/religiao/papa-francisco...,0.0
1,Equador prepara cova coletiva para mortos por ...,NaN,o governar equador anunciar preparar cova cole...,saúde,27/03/2020 18h25,27/03/2020 18h25,https://noticias.uol.com.br/internacional/ulti...,1.0
2,Air France voltará a operar voo direto Pequim-...,NaN,o companhia air france operar voar direto pequ...,saúde,07/08/2020 13h42,07/08/2020 13h42,https://www.uol.com.br/nossa/noticias/afp/2020...,1.0
3,Marfrig intensifica venda de carne do Brasil a...,NaN,o marfrig global foods retomar vender carnar b...,saúde,27/04/2020 14h53,27/04/2020 14h53,https://economia.uol.com.br/noticias/reuters/2...,1.0
4,As parciais das eleições de 2014 alternaram ma...,NaN,o assunto voltar o compartilhar rede social ju...,entretenimento,31/07/2021,Gilmar Lopes,https://www.e-farsas.com/as-parciais-das-eleic...,0.0


In [4]:
print(df["Classe"].value_counts())
print()
print(df.isna().sum())

Classe
0.0    5951
1.0    5951
Name: count, dtype: int64

Titulo         31
Subtitulo    6323
Noticia         1
Categoria       1
Data          352
Autor          17
URL             1
Classe          1
dtype: int64


In [5]:
print("Notícias duplicadas:", df.duplicated(subset="Noticia").sum())

Notícias duplicadas: 16


In [6]:
df["n_palavras"] = df["Noticia"].astype(str).str.split().str.len()
print(df.groupby("Classe")["n_palavras"].describe())

         count        mean        std  min   25%   50%    75%     max
Classe                                                               
0.0     5951.0   68.551168  54.302842  1.0  35.0  61.0   85.0  1233.0
1.0     5951.0  101.896992  73.318375  1.0  70.0  89.0  113.0  1698.0


In [7]:
df["dominio"] = df["URL"].astype(str).str.extract(r"https?://(?:www\.)?([^/]+)")[0]
print(pd.crosstab(df["dominio"], df["Classe"]).sort_values(0.0, ascending=False).head(15))
print()
print(pd.crosstab(df["Categoria"], df["Classe"]))

Classe                      0.0   1.0
dominio                              
boatos.org                 2484     0
g1.globo.com               1184  1100
e-farsas.com                804     0
noticias.uol.com.br         588  3437
checamos.afp.com            503     0
projetocomprova.com.br      388     0
economia.uol.com.br           0   312
educacao.uol.com.br           0    43
entretenimento.uol.com.br     0    65
extra.globo.com               0   223
gov.br                        0   215
tab.uol.com.br                0     4
tvefamosos.uol.com.br         0    64
uol.com.br                    0   488

Classe           0.0   1.0
Categoria                 
brasil           830    74
ciência          185   417
entretenimento  1409     0
mundo            576     4
política        1815  2136
saúde           1136  3320


In [8]:
print("FALSA (0):\n", df[df["Classe"] == 0]["Noticia"].iloc[0][:500])
print()
print("VERDADEIRA (1):\n", df[df["Classe"] == 1]["Noticia"].iloc[0][:500])

FALSA (0):
 apagão vaticano papar presar acusação tráfico criança e fraude .. o papar francisco tuitou manhã o curador beaver confirmar o papar francisco equipar rede social agendar publicação e planejada antecedência clicar tweet o tweetdeck agendar o publicação.o papar francisco custódio federal itália vestir vídeo o fbi o interrogatório fontes.oficiais militar policiar italiano e unidade crime sexual o casar papar vaticano prender e policiar alto e colocar prisão incidente pessoa relatar ouvir tiro o pol

VERDADEIRA (1):
 o governar equador anunciar preparar cova coletiva cemitério enterrar morto coronavírus cidade guaiaquil país divulgar o sepultura ficar pronto o redar ecuavisa sepulcro construir homenagem s vítima o governar federal negociar o prefeitura quaiaquil o cova o manhã hoje o cidade registrar caso confirmar covid-19 o município atingir vírus o país ministrar interior maría paula romo declarar entrevisto redar agência funerário trabalhar disponível função tocar recolher 

In [9]:
# fica só com o que vamos usar
dados = df[["Noticia", "Classe"]].copy()

# remove a linha lixo e as duplicatas
dados = dados.dropna(subset=["Noticia", "Classe"])
dados = dados.drop_duplicates(subset="Noticia")

# remove textos muito curtos (menos de 10 palavras)
dados["Noticia"] = dados["Noticia"].astype(str)
dados["n_palavras"] = dados["Noticia"].str.split().str.len()
print("Textos com menos de 10 palavras:", (dados["n_palavras"] < 10).sum())
dados = dados[dados["n_palavras"] >= 10]

dados["Classe"] = dados["Classe"].astype(int)
print(dados.shape)
print(dados["Classe"].value_counts())

Textos com menos de 10 palavras: 102
(11784, 3)
Classe
1    5932
0    5852
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X = dados["Noticia"]
y = dados["Classe"]

# 70% treino, 30% para dividir em validação e teste
X_treino, X_temp, y_treino, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
# os 30% se dividem ao meio: 15% validação, 15% teste
X_val, X_teste, y_val, y_teste = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("treino:", len(X_treino), "| validação:", len(X_val), "| teste:", len(X_teste))
print(y_treino.value_counts(normalize=True))
print(y_teste.value_counts(normalize=True))

treino: 8248 | validação: 1768 | teste: 1768
Classe
1    0.503395
0    0.496605
Name: proportion, dtype: float64
Classe
1    0.503394
0    0.496606
Name: proportion, dtype: float64


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Passo 1: o vetorizador aprende o vocabulário SÓ com o treino
vetorizador = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
Xtr = vetorizador.fit_transform(X_treino)

# aplica o mesmo vocabulário na validação e no teste (sem reaprender)
Xva = vetorizador.transform(X_val)
Xte = vetorizador.transform(X_teste)

print("formato da matriz de treino:", Xtr.shape)

# Passo 2: treina o modelo
modelo = LogisticRegression(max_iter=1000)
modelo.fit(Xtr, y_treino)

print("treino terminou")

formato da matriz de treino: (8248, 30000)
treino terminou


In [12]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

pred_treino = modelo.predict(Xtr)
pred_val = modelo.predict(Xva)

print("Acurácia treino:    ", round(accuracy_score(y_treino, pred_treino), 4))
print("Acurácia validação: ", round(accuracy_score(y_val, pred_val), 4))
print()
print(classification_report(y_val, pred_val, target_names=["falsa (0)", "verdadeira (1)"]))
print("Matriz de confusão (linhas = real, colunas = previsto):")
print(confusion_matrix(y_val, pred_val))

Acurácia treino:     0.9709
Acurácia validação:  0.9321

                precision    recall  f1-score   support

     falsa (0)       0.93      0.93      0.93       878
verdadeira (1)       0.93      0.93      0.93       890

      accuracy                           0.93      1768
     macro avg       0.93      0.93      0.93      1768
  weighted avg       0.93      0.93      0.93      1768

Matriz de confusão (linhas = real, colunas = previsto):
[[819  59]
 [ 61 829]]


In [13]:
import numpy as np

palavras = np.array(vetorizador.get_feature_names_out())
pesos = modelo.coef_[0]          # positivo = puxa para "verdadeira", negativo = puxa para "falsa"

ordem = np.argsort(pesos)
print("Palavras que mais indicam FALSA:")
print(list(palavras[ordem[:30]]))
print()
print("Palavras que mais indicam VERDADEIRA:")
print(list(palavras[ordem[-30:]][::-1]))

Palavras que mais indicam FALSA:
['circular', 'circular rede', 'compartilhar', 'mensagem', 'umar', 'pra', 'vídeo', 'parir', 'rede social', 'rede', 'texto', 'pelar', 'facebook', 'verdade', 'foto', 'fake', 'site', 'falso', 'publicação', 'lula', 'imagem', '00', 'checar', 'comer', 'olhar', 'postagens', 'whatsapp', 'sobrar', 'fato', 'tá']

Palavras que mais indicam VERDADEIRA:
['feira', 'hoje', 'coronavírus', 'de', 'saudar', 'pandemia', 'caso', 'ministério', 'decisão', 'país', 'casar', 'pandemia coronavírus', 'que', 'ler íntegro', 'íntegro', 'presidente', 'segunda', 'jair bolsonaro', 'pedir', 'segunda feira', 'do', 'covid 19', 'jair', 'contar', 'medir', '19', 'quarta', 'quarta feira', 'uol', 'público']


In [14]:
val = X_val.to_frame("texto")
val["real"] = y_val.values
val["previsto"] = pred_val
val["n_palavras"] = val["texto"].str.split().str.len()
val["acertou"] = val["real"] == val["previsto"]
val["faixa"] = pd.qcut(val["n_palavras"], 4)

print(val.groupby("faixa", observed=True)["acertou"].mean())

faixa
(9.999, 53.0]      0.925110
(53.0, 76.0]       0.923611
(76.0, 101.0]      0.930180
(101.0, 1092.0]    0.949772
Name: acertou, dtype: float64


In [15]:
from sklearn.linear_model import LogisticRegression

tam_tr = X_treino.str.split().str.len().values.reshape(-1, 1)
tam_va = X_val.str.split().str.len().values.reshape(-1, 1)

bobo = LogisticRegression().fit(tam_tr, y_treino)
print("Acurácia usando SÓ o tamanho do texto:", round(bobo.score(tam_va, y_val), 4))

Acurácia usando SÓ o tamanho do texto: 0.6686


In [16]:
atalhos = ["uol", "ler", "íntegro", "feira", "segunda", "terça", "quarta", "quinta",
           "sexta", "hoje", "circular", "compartilhar", "mensagem", "rede", "social",
           "facebook", "whatsapp", "fake", "falso", "checar", "fato", "verdade", "site",
           "publicação", "postagens", "vídeo", "foto", "imagem", "pra", "tá", "umar", "texto"]

vet2 = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2, stop_words=atalhos)
Xtr2 = vet2.fit_transform(X_treino)
Xva2 = vet2.transform(X_val)

modelo2 = LogisticRegression(max_iter=1000).fit(Xtr2, y_treino)
print("Validação sem atalhos:", round(modelo2.score(Xva2, y_val), 4))

Validação sem atalhos: 0.9033


In [17]:
dados["dominio"] = df.loc[dados.index, "dominio"]

fontes_teste = ["e-farsas.com", "checamos.afp.com",                     # falsas
                "uol.com.br", "economia.uol.com.br", "extra.globo.com", "gov.br"]  # verdadeiras
eh_teste = dados["dominio"].isin(fontes_teste)

tr, te = dados[~eh_teste], dados[eh_teste]
print("treino:", len(tr), "| teste (fontes novas):", len(te))
print(te["Classe"].value_counts())

vet3 = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
m3 = LogisticRegression(max_iter=1000).fit(vet3.fit_transform(tr["Noticia"]), tr["Classe"])

pred3 = m3.predict(vet3.transform(te["Noticia"]))
print(classification_report(te["Classe"], pred3, target_names=["falsa (0)", "verdadeira (1)"]))

treino: 9320 | teste (fontes novas): 2464
Classe
1    1235
0    1229
Name: count, dtype: int64
                precision    recall  f1-score   support

     falsa (0)       0.93      0.96      0.95      1229
verdadeira (1)       0.96      0.93      0.94      1235

      accuracy                           0.94      2464
     macro avg       0.94      0.94      0.94      2464
  weighted avg       0.94      0.94      0.94      2464



In [18]:
te = te.copy()
te["previsto"] = pred3
te["acertou"] = te["Classe"] == te["previsto"]

print("Acurácia por fonte:")
print(te.groupby("dominio")["acertou"].agg(["mean", "count"]))
print()

erros = te[~te["acertou"]]
for _, linha in erros.sample(5, random_state=1).iterrows():
    print("REAL:", linha["Classe"], "| PREVISTO:", linha["previsto"], "|", linha["dominio"])
    print(linha["Noticia"][:300])
    print("---")

Acurácia por fonte:
                         mean  count
dominio                             
checamos.afp.com     0.936382    503
e-farsas.com         0.972452    726
economia.uol.com.br  0.964744    312
extra.globo.com      0.900901    222
gov.br               0.985981    214
uol.com.br           0.899384    487

REAL: 1 | PREVISTO: 0 | uol.com.br
decisão tido inédito o justiçar trabalhar rir conceder liminar o sindicato determinar o redar globo depositar r$ relativo o salário atrasar empregar confederação brasileiro desporto aquático cbda especialista ouvir olhar olímpico o decisão abrir atleta e funcionário clube especialmente série b cobri
---
REAL: 0 | PREVISTO: 1 | e-farsas.com
o blog haver intenção ferir o honrar haver ano vir lutar igualdade racial cidade
---
REAL: 0 | PREVISTO: 1 | checamos.afp.com
“ comandante corpo fuzileiro naval repreender pelosi ‘ trabalhar ’ general recusar o pedir presidente câmara nancy pelosi marine segurança posse joe biden ” o título e subtítulo ar

In [19]:
df["data_limpa"] = pd.to_datetime(df["Data"].astype(str).str.extract(r"(\d{2}/\d{2}/\d{4})")[0],
                                  format="%d/%m/%Y", errors="coerce")
print("sem data reconhecida:", df["data_limpa"].isna().sum())
print(df["data_limpa"].min(), "->", df["data_limpa"].max())
print()
print(df.groupby([df["data_limpa"].dt.year, "Classe"]).size().unstack())

sem data reconhecida: 1360
2018-08-29 00:00:00 -> 2021-08-31 00:00:00

Classe         0.0     1.0
data_limpa                
2018.0        94.0     NaN
2019.0      1365.0     8.0
2020.0      2293.0  3980.0
2021.0      1063.0  1740.0


In [20]:
dados["data_limpa"] = df.loc[dados.index, "data_limpa"]
dados["ano"] = dados["data_limpa"].dt.year

print("Classe das notícias SEM data:")
print(dados[dados["ano"].isna()]["Classe"].value_counts())
print()

val = X_val.to_frame("texto")
val["real"] = y_val.values
val["previsto"] = modelo.predict(Xva)
val["ano"] = dados.loc[X_val.index, "ano"].values
val["acertou"] = val["real"] == val["previsto"]
print(val.groupby("ano", dropna=False)["acertou"].agg(["mean", "count"]))

Classe das notícias SEM data:
Classe
0    1100
1     222
Name: count, dtype: int64

            mean  count
ano                    
2018.0  1.000000      9
2019.0  0.947115    208
2020.0  0.922009    936
2021.0  0.950588    425
NaN     0.921053    190


In [21]:
pool = dados.loc[X_treino.index.union(X_val.index)]   # sem tocar no teste
pool = pool[pool["ano"].isin([2020, 2021])]
tr, te = pool[pool["ano"] == 2020], pool[pool["ano"] == 2021]
print("treino 2020:", len(tr), "| teste 2021:", len(te))

vet_t = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
m_t = LogisticRegression(max_iter=1000).fit(vet_t.fit_transform(tr["Noticia"]), tr["Classe"])
pred_t = m_t.predict(vet_t.transform(te["Noticia"]))
print(classification_report(te["Classe"], pred_t, target_names=["falsa (0)", "verdadeira (1)"]))

treino 2020: 5307 | teste 2021: 2392
                precision    recall  f1-score   support

     falsa (0)       0.95      0.81      0.88       914
verdadeira (1)       0.89      0.98      0.93      1478

      accuracy                           0.91      2392
     macro avg       0.92      0.89      0.90      2392
  weighted avg       0.92      0.91      0.91      2392



In [22]:
m_b = LogisticRegression(max_iter=1000, class_weight="balanced")
m_b.fit(vet_t.transform(tr["Noticia"]), tr["Classe"])
pred_b = m_b.predict(vet_t.transform(te["Noticia"]))
print(classification_report(te["Classe"], pred_b, target_names=["falsa (0)", "verdadeira (1)"]))
print(confusion_matrix(te["Classe"], pred_b))

                precision    recall  f1-score   support

     falsa (0)       0.92      0.89      0.90       914
verdadeira (1)       0.93      0.95      0.94      1478

      accuracy                           0.93      2392
     macro avg       0.92      0.92      0.92      2392
  weighted avg       0.93      0.93      0.93      2392

[[ 812  102]
 [  73 1405]]


In [23]:
import re

amostra = dados["Noticia"]
print("textos com 'http' ou 'www':", amostra.str.contains(r"http|www", regex=True).sum())
print("textos com números de data (dd/mm/aaaa):", amostra.str.contains(r"\d{1,2}/\d{1,2}/\d{2,4}").sum())
print("textos com 2019/2020/2021:", amostra.str.contains(r"\b(2019|2020|2021)\b").sum())
print("textos com 'uol':", amostra.str.contains("uol").sum())

textos com 'http' ou 'www': 16
textos com números de data (dd/mm/aaaa): 142
textos com 2019/2020/2021: 289
textos com 'uol': 437


/var/folders/jf/thxnqvlj4hxc5c6lw0gc8cq00000gp/T/ipykernel_3358/2257638979.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("textos com 2019/2020/2021:", amostra.str.contains(r"\b(2019|2020|2021)\b").sum())


In [24]:
def limpar(t):
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"\d{1,2}/\d{1,2}/\d{2,4}", " ", t)
    t = re.sub(r"\d{1,2}h\d{0,2}", " ", t)
    t = re.sub(r"\b(uol|globo|g1|afp|extra|boatos|e-farsas)\b", " ", t)
    return re.sub(r"\s+", " ", t).strip()

Xtr_l = X_treino.apply(limpar)
Xva_l = X_val.apply(limpar)

vet_l = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
m_l = LogisticRegression(max_iter=1000).fit(vet_l.fit_transform(Xtr_l), y_treino)
print("Validação com texto limpo:", round(m_l.score(vet_l.transform(Xva_l), y_val), 4))
print("Validação sem limpar:     0.9321")

Validação com texto limpo: 0.9321
Validação sem limpar:     0.9321


In [25]:
mudou = (X_treino != Xtr_l).sum()
print("textos do treino alterados pela limpeza:", mudou, "de", len(X_treino))
print("previsões diferentes na validação:",
      (modelo.predict(Xva) != m_l.predict(vet_l.transform(Xva_l))).sum())

textos do treino alterados pela limpeza: 1300 de 8248
previsões diferentes na validação: 8


In [26]:
df2 = pd.read_excel("dados/FakeRecogna-master/dataset/FakeRecogna_no_removal_words.xlsx")
print(df2.shape)
print(df2.columns.tolist())
print(df2["Noticia"].iloc[0][:400])
print()
print(df2["Noticia"].iloc[1][:400])

(11902, 8)
['Titulo', 'Subtitulo', 'Noticia', 'Categoria', 'Data', 'Autor', 'URL', 'Classe']
bomba lira fraudar o votação brasília explodir votar impresso auditável perder roubar umar haver fraudar o votação o câmara deputar imaginar o eleição presidencial bomba bomba bomba bomba lira fraudar o votação brasília explodir

o whatsapp liberar quinta-feira atualização aplicativo celular android e iphones permitir silenciar conversar e abaixar opção testar versão betar antecipar recurso instável o novidade liberar atualização o o selecionar o opção silenciar conversar o opção deixar notificação mudar ano substituir “ ” “ indeterminar ” opção horar semana continuar disponível


In [27]:
d2 = df2[["Noticia", "Classe"]].dropna(subset=["Noticia", "Classe"]).copy()
d2 = d2.drop_duplicates(subset="Noticia")
d2["Noticia"] = d2["Noticia"].astype(str)
d2 = d2[d2["Noticia"].str.split().str.len() >= 10]
d2["Classe"] = d2["Classe"].astype(int)
print(d2.shape)

X2, y2 = d2["Noticia"], d2["Classe"]
Xt2, Xtmp2, yt2, ytmp2 = train_test_split(X2, y2, test_size=0.30, stratify=y2, random_state=42)
Xv2, Xte2, yv2, yte2 = train_test_split(Xtmp2, ytmp2, test_size=0.50, stratify=ytmp2, random_state=42)

vet2b = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
m2 = LogisticRegression(max_iter=1000).fit(vet2b.fit_transform(Xt2), yt2)
print("Validação (no_removal_words):", round(m2.score(vet2b.transform(Xv2), yv2), 4))
print("Validação (versão anterior): 0.9321")

(11784, 2)
Validação (no_removal_words): 0.9536
Validação (versão anterior): 0.9321


In [28]:
# 1) palavras mais decisivas na nova versão
import numpy as np
pal = np.array(vet2b.get_feature_names_out())
w = m2.coef_[0]; o = np.argsort(w)
print("FALSA:", list(pal[o[:30]]))
print("VERDADEIRA:", list(pal[o[-30:]][::-1]))

# 2) treinar em 2020, avaliar em 2021, com class_weight balanceado
d2["ano"] = dados["data_limpa"].dt.year.reindex(d2.index) if len(d2) == len(dados) else None

FALSA: ['rede', 'rede social', 'circular', 'compartilhar', 'circular rede', 'vídeo', 'pra', 'mensagem', 'facebook', 'falso', 'texto', 'chino', 'feito', 'verdade', 'site', 'publicação', 'amigo', 'foto', 'fake', 'whatsapp', 'social', '00', 'checar', 'lula', 'filho', 'imagem', 'tá', 'esquerdo', 'espalhar', 'urgente']
VERDADEIRA: ['feira', 'coronavírus', 'hoje', 'casar', 'pandemia', 'saudar', 'dosar', 'país', 'ministrar', 'pandemia coronavírus', 'entrevistar', 'decisão', 'candidatar', 'ministério', 'íntegro', 'nacional', 'redar social', 'ler íntegro', 'precisar', 'presidente', 'chinar', 'jair bolsonaro', 'unir', 'paulo', 'jair', 'medir', 'quinta', 'quinta feira', 'passar', 'covid 19']


In [29]:
import os
os.makedirs("dados/export", exist_ok=True)

pd.DataFrame({"texto": Xt2, "label": yt2}).to_csv("dados/export/treino.csv", index=False)
pd.DataFrame({"texto": Xv2, "label": yv2}).to_csv("dados/export/validacao.csv", index=False)
pd.DataFrame({"texto": Xte2, "label": yte2}).to_csv("dados/export/teste.csv", index=False)  # guardado, não usar ainda
print("ok")

ok


In [30]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

vet_c = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
A_tr = vet_c.fit_transform(Xt2)
A_va = vet_c.transform(Xv2)

candidatos = {
    "Regressão Logística": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "SVM (linear)": LinearSVC(),
}

for nome_m, m in candidatos.items():
    m.fit(A_tr, yt2)
    pred = m.predict(A_va)
    print("=" * 50)
    print(nome_m, "| acurácia validação:", round(m.score(A_va, yv2), 4))
    print(classification_report(yv2, pred, target_names=["falsa (0)", "verdadeira (1)"], digits=3))

Regressão Logística | acurácia validação: 0.9536
                precision    recall  f1-score   support

     falsa (0)      0.956     0.950     0.953       878
verdadeira (1)      0.951     0.957     0.954       890

      accuracy                          0.954      1768
     macro avg      0.954     0.954     0.954      1768
  weighted avg      0.954     0.954     0.954      1768

Naive Bayes | acurácia validação: 0.9338
                precision    recall  f1-score   support

     falsa (0)      0.952     0.912     0.932       878
verdadeira (1)      0.917     0.955     0.936       890

      accuracy                          0.934      1768
     macro avg      0.935     0.934     0.934      1768
  weighted avg      0.935     0.934     0.934      1768

SVM (linear) | acurácia validação: 0.9615
                precision    recall  f1-score   support

     falsa (0)      0.969     0.953     0.961       878
verdadeira (1)      0.955     0.970     0.962       890

      accuracy      

In [31]:
m_svm = candidatos["SVM (linear)"]
print(classification_report(yv2, m_svm.predict(A_va), target_names=["falsa (0)", "verdadeira (1)"], digits=3))

                precision    recall  f1-score   support

     falsa (0)      0.969     0.953     0.961       878
verdadeira (1)      0.955     0.970     0.962       890

      accuracy                          0.962      1768
     macro avg      0.962     0.961     0.962      1768
  weighted avg      0.962     0.962     0.962      1768



In [32]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import sklearn

modelo_final = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)),
    ("clf", LinearSVC()),
])
modelo_final.fit(Xt2, yt2)

print("Validação:", round(modelo_final.score(Xv2, yv2), 4))   # deve dar 0.9615

# ===== TESTE FINAL: rode esta parte UMA vez =====
pred_teste = modelo_final.predict(Xte2)
print("Teste:", round(modelo_final.score(Xte2, yte2), 4))
print(classification_report(yte2, pred_teste, target_names=["falsa (0)", "verdadeira (1)"], digits=3))
print(confusion_matrix(yte2, pred_teste))
print("scikit-learn:", sklearn.__version__)

Validação: 0.9615
Teste: 0.957
                precision    recall  f1-score   support

     falsa (0)      0.961     0.952     0.957       878
verdadeira (1)      0.953     0.962     0.957       890

      accuracy                          0.957      1768
     macro avg      0.957     0.957     0.957      1768
  weighted avg      0.957     0.957     0.957      1768

[[836  42]
 [ 34 856]]
scikit-learn: 1.9.1


In [33]:
import joblib

joblib.dump(modelo_final, "modelo_fakerecogna_svm.pkl")

modelo = joblib.load("modelo_fakerecogna_svm.pkl")
exemplo = ["o whatsapp liberar quinta-feira atualização aplicativo permitir silenciar conversar"]
print("Previsão (0 = falsa, 1 = verdadeira):", modelo.predict(exemplo))
print("Distância da fronteira:", modelo.decision_function(exemplo))

Previsão (0 = falsa, 1 = verdadeira): [1]
Distância da fronteira: [0.83254229]
